# Claude API - Примеры использования

Этот ноутбук демонстрирует работу с Claude API через Anthropic SDK.

**Документация:** https://docs.anthropic.com/en/docs

## 1. Настройка API ключа

Получить ключ можно на https://console.anthropic.com/

**Важно:** Никогда не храни ключ в коде! Используй переменные окружения.

In [ ]:
import os
from anthropic import Anthropic

# Вариант 1: Из переменной окружения (рекомендуется)
# Установи: set ANTHROPIC_API_KEY=sk-ant-...
# client = Anthropic()  # автоматически берёт из ANTHROPIC_API_KEY

# Вариант 2: Напрямую (только для тестов!)
# Замени на свой ключ или установи переменную окружения
api_key = os.getenv('ANTHROPIC_API_KEY', 'YOUR_API_KEY_HERE')

if api_key == 'YOUR_API_KEY_HERE':
    print('⚠️ Установи API ключ!')
    print('В терминале: set ANTHROPIC_API_KEY=sk-ant-...')
    print('Или замени YOUR_API_KEY_HERE на свой ключ') 
else:
    client = Anthropic(api_key=api_key)
    print('✅ API ключ установлен')

## 2. Простой запрос к Claude

In [ ]:
# Базовый запрос
message = client.messages.create(
    model="claude-sonnet-4-20250514",  # или claude-3-haiku-20240307 (дешевле)
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Привет! Объясни в 2 предложениях, что такое машинное обучение."}
    ]
)

print(message.content[0].text)

## 3. Системный промпт

Системный промпт задаёт роль и поведение модели.

In [ ]:
message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="Ты - опытный Python разработчик. Отвечай кратко и с примерами кода.",
    messages=[
        {"role": "user", "content": "Как отсортировать список словарей по ключу?"}
    ]
)

print(message.content[0].text)

## 4. Диалог (Multi-turn conversation)

Claude помнит контекст разговора, если передавать историю сообщений.

In [ ]:
# История диалога
conversation = [
    {"role": "user", "content": "Меня зовут Руслан, я изучаю машинное обучение."},
    {"role": "assistant", "content": "Приятно познакомиться, Руслан! Машинное обучение - отличная область. Чем именно ты сейчас занимаешься?"},
    {"role": "user", "content": "Как меня зовут и что я изучаю?"}
]

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=256,
    messages=conversation
)

print(message.content[0].text)

## 5. Параметры генерации

In [ ]:
# temperature - креативность (0 = детерминированный, 1 = творческий)
# max_tokens - максимум токенов в ответе
# top_p - nucleus sampling

# Пример: творческий ответ
creative = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    temperature=0.9,  # высокая креативность
    messages=[{"role": "user", "content": "Придумай название для стартапа по AI-автоматизации"}]
)

print("Творческий (temp=0.9):")
print(creative.content[0].text)

# Пример: точный ответ
precise = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    temperature=0,  # детерминированный
    messages=[{"role": "user", "content": "Сколько будет 15 * 17?"}]
)

print("\nТочный (temp=0):")
print(precise.content[0].text)

## 6. Streaming (потоковый вывод)

Получение ответа по частям - полезно для длинных ответов.

In [ ]:
print("Ответ появляется по частям:")
print("-" * 40)

with client.messages.stream(
    model="claude-sonnet-4-20250514",
    max_tokens=500,
    messages=[{"role": "user", "content": "Напиши короткую историю про робота в 3 абзаца."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print("\n" + "-" * 40)

## 7. Структурированный вывод (JSON)

Полезно для интеграции в приложения.

In [ ]:
import json

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=500,
    system="Отвечай только валидным JSON без markdown.",
    messages=[{
        "role": "user", 
        "content": """Проанализируй текст и верни JSON:
        {
            "sentiment": "positive/negative/neutral",
            "topics": ["тема1", "тема2"],
            "summary": "краткое содержание"
        }
        
        Текст: Сегодня отличный день! Я выучил новую технологию и сделал крутой проект.
        """
    }]
)

result = json.loads(message.content[0].text)
print("Тональность:", result['sentiment'])
print("Темы:", result['topics'])
print("Резюме:", result['summary'])

## 8. Работа с изображениями (Vision)

Claude может анализировать изображения.

In [ ]:
import base64
import httpx

# Загрузка изображения по URL
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg"
image_data = base64.standard_b64encode(httpx.get(image_url).content).decode("utf-8")

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=500,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": image_data,
                    },
                },
                {
                    "type": "text",
                    "text": "Что изображено на картинке? Опиши подробно."
                }
            ],
        }
    ],
)

print(message.content[0].text)

## 9. Подсчёт токенов и стоимости

In [ ]:
# Информация о последнем запросе
print(f"Модель: {message.model}")
print(f"Входные токены: {message.usage.input_tokens}")
print(f"Выходные токены: {message.usage.output_tokens}")

# Примерные цены (актуальные на docs.anthropic.com)
# Claude 3.5 Sonnet: $3 / 1M input, $15 / 1M output
# Claude 3 Haiku: $0.25 / 1M input, $1.25 / 1M output

input_cost = message.usage.input_tokens * 3 / 1_000_000
output_cost = message.usage.output_tokens * 15 / 1_000_000
total = input_cost + output_cost

print(f"\nПримерная стоимость запроса: ${total:.6f}")

## 10. Обработка ошибок

In [ ]:
from anthropic import APIError, RateLimitError, AuthenticationError

def safe_request(prompt):
    try:
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    
    except AuthenticationError:
        return "Ошибка: неверный API ключ"
    
    except RateLimitError:
        return "Ошибка: превышен лимит запросов, подожди немного"
    
    except APIError as e:
        return f"Ошибка API: {e}"

# Тест
result = safe_request("Скажи 'Привет!'")
print(result)

## 11. Практический пример: Анализатор кода

In [ ]:
def analyze_code(code: str) -> dict:
    """Анализирует код и возвращает рекомендации."""
    
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        system="""Ты - code reviewer. Анализируй код и возвращай JSON:
        {
            "quality": 1-10,
            "issues": ["проблема1", "проблема2"],
            "suggestions": ["улучшение1", "улучшение2"],
            "improved_code": "улучшенный код"
        }
        Отвечай только JSON без markdown.""",
        messages=[{"role": "user", "content": f"Проанализируй код:\n```\n{code}\n```"}]
    )
    
    return json.loads(message.content[0].text)

# Тестовый код для анализа
test_code = '''
def calc(x,y):
    z=x+y
    return z
'''

analysis = analyze_code(test_code)
print(f"Качество: {analysis['quality']}/10")
print(f"\nПроблемы:")
for issue in analysis['issues']:
    print(f"  - {issue}")
print(f"\nПредложения:")
for suggestion in analysis['suggestions']:
    print(f"  - {suggestion}")
print(f"\nУлучшенный код:")
print(analysis['improved_code'])

## 12. Доступные модели

| Модель | ID | Описание |
|--------|-----|----------|
| Claude 3.5 Sonnet | claude-sonnet-4-20250514 | Лучший баланс цена/качество |
| Claude 3 Opus | claude-3-opus-20240229 | Самый умный, дорогой |
| Claude 3 Haiku | claude-3-haiku-20240307 | Быстрый и дешёвый |

**Совет:** Для экспериментов используй Haiku, для продакшена - Sonnet.

## Следующие шаги

1. **Tool Use (Function Calling)** - Claude может вызывать твои функции
2. **RAG** - Retrieval Augmented Generation с твоими документами
3. **Агенты** - автономные AI-системы с LangChain/LlamaIndex
4. **Embeddings** - векторные представления текста для поиска